# Note 32: FlashAttention — Forward and Backward in NumPy

## Goal

Understand how to compute **the same attention output and gradients** without
storing the full attention matrix. We will derive the equations, implement them,
and check them using tiny examples you can run on a CPU.

The two central ideas are **online softmax** in forward and **recomputation** in
backward. Tiling makes both work on small pieces of the score matrix.

You need matrix multiplication and basic derivatives. We derive the softmax
gradient rather than assuming you already know it. Note 13 is useful background,
but this notebook does not import anything from it.

### Road map

1. Ordinary attention: shapes, stable softmax, and a dense forward pass.
2. Ordinary backward: derive each gradient from the chain rule.
3. Online softmax: combine blocks without losing the global normalization.
4. FlashAttention forward: turn the recurrence into tiled NumPy code.
5. FlashAttention backward: reconstruct one probability tile at a time.
6. Causal attention, numerical checks, memory accounting, and exercises.

**Scope.** One sequence and one head, with no dropout. This is an educational
implementation of the mathematical algorithm, not a GPU kernel or a benchmark.
The forward loop uses the unnormalized accumulator and final normalization
described in FlashAttention-2. We omit GPU scheduling and parallelism.

## Setup

Only **NumPy** is needed by the code cells. Use a Python 3 Jupyter kernel with
NumPy installed, then **Restart Kernel and Run All Cells**. There are no datasets
to download, local helper modules, or training loops.

We use `float64` and fixed random seeds to make small numerical comparisons easy
to interpret. Floating-point operations can round differently when their order
changes, so “same result” means agreement within a stated tolerance.

In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(32)
print("NumPy version:", np.__version__)

NumPy version: 2.3.5


## 1. Ordinary attention: the reference computation

Let $N$ be sequence length, $d$ the query/key dimension, and $d_v$ the value
dimension. Values need not have the same feature dimension as queries and keys.

| Symbol | Shape | Meaning |
|---|---|---|
| $Q$ | $(N,d)$ | One query vector per token |
| $K$ | $(N,d)$ | One key vector per token |
| $V$ | $(N,d_v)$ | One value vector per token |
| $S$ | $(N,N)$ | Scaled query–key scores |
| $P$ | $(N,N)$ | Attention probabilities, normalized across keys |
| $O$ | $(N,d_v)$ | Weighted values, one output per query |

$$
S=\frac{QK^\top}{\sqrt d},\qquad
P_{ij}=\frac{\exp(S_{ij})}{\sum_t\exp(S_{it})},\qquad O=PV.
$$

Row $i$ of $P$ answers: **which keys does query $i$ attend to?** Thus
$O_i=\sum_jP_{ij}V_j$. The factor $1/\sqrt d$ controls the score scale as the
feature dimension grows. It must also appear in the gradients of $Q$ and $K$.

### Stable softmax

Exponentiating a large score can overflow. Subtracting the row maximum does not
change the probabilities, because the common factor cancels:

$$
m_i=\max_j S_{ij},\qquad
P_{ij}=\frac{\exp(S_{ij}-m_i)}{\sum_t\exp(S_{it}-m_i)}.
$$

In NumPy, `axis=1` reduces across keys and `keepdims=True` retains shape $(N,1)$
so subtraction and division broadcast across each row.

In [2]:
def softmax_rows(scores):
    """Normalize each row; each row must contain at least one finite score."""
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    weights = np.exp(shifted)
    return weights / np.sum(weights, axis=1, keepdims=True)


def attention_forward(Q, K, V, causal=False):
    """Dense single-head attention. Return output O and saved probabilities P."""
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    if causal:
        positions = np.arange(Q.shape[0])
        allowed = positions[:, None] >= positions[None, :]
        scores = np.where(allowed, scores, -np.inf)
    P = softmax_rows(scores)
    O = P @ V
    return O, P

### A tiny example you can inspect

Initially use `causal=False`: every query may attend to every key. The optional
causal branch is explained in module 6. The inputs below are fixed so that you
can compare the score, probability, and output rows directly.

In [3]:
Q_tiny = np.array([[1., 0.], [0., 1.], [1., 1.]])
K_tiny = np.array([[1., 0.], [0., 1.], [-1., 1.]])
V_tiny = np.array([[1., 2.], [3., 0.], [0., 4.]])

O_tiny, P_tiny = attention_forward(Q_tiny, K_tiny, V_tiny)
print("Scores:\n", Q_tiny @ K_tiny.T / np.sqrt(2))
print("Probabilities:\n", P_tiny)
print("Row sums:", P_tiny.sum(axis=1))
print("Output:\n", O_tiny)
np.testing.assert_allclose(P_tiny.sum(axis=1), 1.0)
np.testing.assert_allclose(O_tiny[0], sum(P_tiny[0, j] * V_tiny[j] for j in range(3)))

# Hand-checkable special case: equal scores give an ordinary mean of V.
uniform_O, _ = attention_forward(np.zeros_like(Q_tiny), K_tiny, V_tiny)
np.testing.assert_allclose(uniform_O, np.tile(V_tiny.mean(axis=0), (3, 1)))
print("Equal-score check passed: each output is the mean value.")

Scores:
 [[ 0.70711  0.      -0.70711]
 [ 0.       0.70711  0.70711]
 [ 0.70711  0.70711  0.     ]]
Probabilities:
 [[0.57598 0.284   0.14003]
 [0.19778 0.40111 0.40111]
 [0.40111 0.40111 0.19778]]
Row sums: [1. 1. 1.]
Output:
 [[1.42796 1.71207]
 [1.40111 2.     ]
 [1.60445 1.59333]]
Equal-score check passed: each output is the mean value.


### Where the quadratic memory comes from

This reference creates all $N^2$ scores and all $N^2$ probabilities. It keeps
$P$ for backward. At long sequence lengths, these arrays can be much larger
than the inputs, whose sizes grow like $Nd$ or $Nd_v$.

FlashAttention still considers every allowed query–key pair. Its improvement is
to compute and discard small score/probability blocks, while retaining enough
information to combine them correctly and reconstruct them later.

## 2. Ordinary backward, one derivative at a time

Let $\mathcal L$ be any scalar loss depending on $O$. We receive

$$G=\frac{\partial\mathcal L}{\partial O}\in\mathbb R^{N\times d_v}.$$

We will write `dQ`, `dK`, and so on for gradients, **not** infinitesimal changes
and **not** the feature dimension $d$. Our job is to compute the gradients of the
three inputs, given $G$. No automatic differentiation is used.

### 2.1 Back through the weighted sum: $O=PV$

For one output element, $O_{ir}=\sum_jP_{ij}V_{jr}$. Apply the chain rule:

$$
\frac{\partial\mathcal L}{\partial V_{jr}}
=\sum_iG_{ir}P_{ij},\qquad
\frac{\partial\mathcal L}{\partial P_{ij}}
=\sum_rG_{ir}V_{jr}.
$$

Therefore:

$$dV=P^\top G\quad(N,d_v),\qquad dP=GV^\top\quad(N,N).$$

Every query using value $j$ contributes to $dV_j$. For $dP_{ij}$, we ask how
changing the weight on value $j$ would change the loss through output row $i$.

### 2.2 Back through softmax: first derive a single row

We now know $dP=GV^\top$: how the loss changes with each attention probability.
The next step is to find $dS$: how the loss changes with each score before
softmax. Since softmax operates independently on each row, we can derive this
for one query and then apply the result to every row.

Fix query row $i$. To keep the notation readable, write $s_j=S_{ij}$ and
$p_j=P_{ij}$ for its scores and probabilities, and call its probability gradient
$u_j$:

$$
u_j\equiv\frac{\partial\mathcal L}{\partial p_j}
=dP_{ij}=\sum_r G_{ir}V_{jr}.
$$

Thus $u$ is simply row $i$ of the gradient from section 2.1. Each $u_j$ measures
how changing the weight on value $j$ would affect the loss. It is the
**upstream gradient** that reaches softmax through the weighted sum $O=PV$.
We will combine it with the derivative of softmax to obtain the score gradient.

For $p_j=\exp(s_j)/Z$ with $Z=\sum_t\exp(s_t)$, the quotient rule gives

$$
\frac{\partial p_j}{\partial s_k}
=\frac{\delta_{jk}\exp(s_j)Z-\exp(s_j)\exp(s_k)}{Z^2}
=p_j(\delta_{jk}-p_k),
$$

where $\delta_{jk}$ is 1 when $j=k$ and 0 otherwise. Changing score $s_k$
affects every probability through the shared denominator. The chain rule
therefore sums the loss contributions through all $p_j$:

$$
\begin{aligned}
\frac{\partial\mathcal L}{\partial s_k}
&=\sum_j\frac{\partial\mathcal L}{\partial p_j}
          \frac{\partial p_j}{\partial s_k}\\
&=\sum_j u_jp_j(\delta_{jk}-p_k)\\
&=u_kp_k-p_k\sum_jp_ju_j\\
&=p_k\left(u_k-\sum_jp_ju_j\right).
\end{aligned}
$$

The quantity $\sum_jp_ju_j$ is one scalar shared by the entire row. Once we
compute it, each score gradient needs only a subtraction and a multiplication.
This avoids forming the full softmax Jacobian
$J=\operatorname{diag}(p)-pp^\top$ to compute $J^\top u$.

Restoring query index $i$ and replacing $u_j$ with $dP_{ij}$ gives the formula
for all rows:

$$
D_i=\sum_jP_{ij}\,dP_{ij},\qquad
dS_{ij}=P_{ij}(dP_{ij}-D_i).
$$

In NumPy, we calculate `D` with shape $(N,1)`, so `dP - D` broadcasts the
subtraction across keys. The stable, shifted softmax has the same derivative:
subtracting a common constant from a row leaves its probabilities unchanged.

The following small check compares this row-wise formula with the full
Jacobian. We use `upstream_p = [0.7, -0.2, 1.1]` as a sample vector $u$ to test
softmax on its own. In the complete attention backward, that vector comes
from `dP[i]`, where `dP = G @ V.T`.


In [4]:
# Check the short formula against the full Jacobian for just one tiny row.
p = P_tiny[0]
upstream_p = np.array([0.7, -0.2, 1.1])
jacobian = np.diag(p) - np.outer(p, p)
via_jacobian = jacobian.T @ upstream_p
via_row_sum = p * (upstream_p - np.sum(p * upstream_p))
np.testing.assert_allclose(via_jacobian, via_row_sum, atol=1e-14)
print("Softmax score gradient:", via_row_sum)
print("Sum of score gradients:", via_row_sum.sum())

Softmax score gradient: [ 0.11496 -0.19891  0.08396]
Sum of score gradients: -1.3877787807814457e-17


The score gradients sum to approximately zero. This matches the fact that
adding a common constant to all scores in a row cannot change the output.

### 2.3 Back through the scores: $S=QK^\top/\sqrt d$

Since $S_{ij}=\sum_aQ_{ia}K_{ja}/\sqrt d$,

$$
dQ_{ia}=\frac{1}{\sqrt d}\sum_jdS_{ij}K_{ja},\qquad
dK_{ja}=\frac{1}{\sqrt d}\sum_idS_{ij}Q_{ia}.
$$

In matrix form:

$$dQ=\frac{dS K}{\sqrt d},\qquad dK=\frac{dS^\top Q}{\sqrt d}.$$

Both have shape $(N,d)$. Gradients add whenever multiple operations depend on
the same input. That accumulation will become explicit when we split the
matrices into tiles.

In [5]:
def attention_backward(Q, K, V, P, G):
    """Dense backward given saved P and upstream gradient G = dL/dO."""
    scale = 1.0 / np.sqrt(Q.shape[1])
    dV = P.T @ G
    dP = G @ V.T
    D = np.sum(P * dP, axis=1, keepdims=True)
    dS = P * (dP - D)
    dQ = scale * (dS @ K)
    dK = scale * (dS.T @ Q)
    return dQ, dK, dV


G_tiny = np.array([[0.2, -0.4], [0.5, 0.3], [-0.1, 0.8]])
dQ_tiny, dK_tiny, dV_tiny = attention_backward(Q_tiny, K_tiny, V_tiny, P_tiny, G_tiny)
for name, gradient in zip(("dQ", "dK", "dV"), (dQ_tiny, dK_tiny, dV_tiny)):
    print(name, gradient.shape, "\n", gradient)

dQ (3, 2) 
 [[ 0.03713  0.08177]
 [ 0.00047  0.02805]
 [-0.18227 -0.10942]]
dK (3, 2) 
 [[ 0.02765  0.08137]
 [-0.20045 -0.34454]
 [ 0.1728   0.26317]]
dV (3, 2) 
 [[0.17397 0.14983]
 [0.21724 0.32763]
 [0.20878 0.22254]]


### 2.4 Check the derivatives independently

Choose $\mathcal L(Q,K,V)=\sum_{ir}O_{ir}G_{ir}$ with **fixed** $G$. Then its
upstream gradient is exactly $G$. For any input entry $x$,

$$
\frac{\partial\mathcal L}{\partial x}
\approx\frac{\mathcal L(x+\varepsilon)-\mathcal L(x-\varepsilon)}{2\varepsilon}.
$$

The next helper perturbs **every element** of $Q,K,V$, not just one randomly
chosen direction. It is slow for large tensors, but our examples are tiny.
Each perturbation calls forward again; it never uses the analytical backward.

In [6]:
def numerical_gradients(forward_output, Q, K, V, G, epsilon=1e-6):
    """Central finite differences for the scalar loss sum(forward(...) * G)."""
    inputs = [Q.copy(), K.copy(), V.copy()]
    gradients = []
    for x in inputs:
        gradient = np.zeros_like(x)
        for index in np.ndindex(x.shape):
            original = x[index]
            x[index] = original + epsilon
            loss_plus = np.sum(forward_output(*inputs) * G)
            x[index] = original - epsilon
            loss_minus = np.sum(forward_output(*inputs) * G)
            x[index] = original
            gradient[index] = (loss_plus - loss_minus) / (2 * epsilon)
        gradients.append(gradient)
    return tuple(gradients)


numeric_tiny = numerical_gradients(
    lambda Q, K, V: attention_forward(Q, K, V)[0],
    Q_tiny, K_tiny, V_tiny, G_tiny,
)
for name, analytic, numeric in zip(
    ("dQ", "dK", "dV"), (dQ_tiny, dK_tiny, dV_tiny), numeric_tiny
):
    np.testing.assert_allclose(analytic, numeric, atol=2e-8, rtol=2e-6)
    print(f"Dense {name}: maximum finite-difference error = {np.max(np.abs(analytic - numeric)):.2e}")

Dense dQ: maximum finite-difference error = 2.17e-10
Dense dK: maximum finite-difference error = 7.96e-11
Dense dV: maximum finite-difference error = 1.97e-10


## 3. Online softmax: combine blocks correctly

### 3.1 Why independently normalizing blocks fails

Suppose a row has scores $[0,0,\log 3,\log 3]$ and scalar values $[0,0,1,1]$.
The unnormalized weights are $[1,1,3,3]$, so the correct output is $6/8=0.75$.

If we normalize each pair separately, their outputs are 0 and 1. Averaging
them gives 0.5. That loses the fact that the second block has **three times the
normalization mass** of the first block.

In [7]:
scores_example = np.array([[0., 0., np.log(3.), np.log(3.)]])
values_example = np.array([[0.], [0.], [1.], [1.]])
correct = (softmax_rows(scores_example) @ values_example).item()
block_outputs = [
    (softmax_rows(scores_example[:, j:j+2]) @ values_example[j:j+2]).item()
    for j in (0, 2)
]
wrong = np.mean(block_outputs)
print(f"Global softmax: {correct:.2f}; average of local outputs: {wrong:.2f}")
np.testing.assert_allclose(correct, 0.75)
assert not np.isclose(correct, wrong)

Global softmax: 0.75; average of local outputs: 0.50


### 3.2 The state of one query after some keys have been processed

For the set of keys already seen, maintain:

$$
m=\max_{j\in\mathrm{seen}}s_j,\qquad
\ell=\sum_{j\in\mathrm{seen}}e^{s_j-m},\qquad
A=\sum_{j\in\mathrm{seen}}e^{s_j-m}V_j.
$$

$m$ and $\ell$ are scalars; $A$ is a vector of length $d_v$. $A$ is a weighted
sum, **not yet a normalized attention output**. At the end, $O=A/\ell$.

For a new block of scores $s_B$ and values $V_B$, choose the new reference maximum

$$m'=\max(m,\max s_B),\qquad \alpha=e^{m-m'}.$$

Why rescale? An old contribution was represented as $e^{s_j-m}$. Measured
relative to $m'$, it is

$$e^{s_j-m'}=e^{s_j-m}\,e^{m-m'}=\alpha e^{s_j-m}.$$

Therefore **both** the old denominator and the old weighted sum must be rescaled:

$$
w_B=e^{s_B-m'},\qquad
\ell'=\alpha\ell+\sum_{j\in B}(w_B)_j,\qquad
A'=\alpha A+w_BV_B.
$$

Initialize $m=-\infty$, $\ell=0$, and $A=0$. On the first finite block,
$\alpha=0$, so there are no old contributions to preserve. These equations
maintain the stated meanings of $m,\ell,A$ after every block, which explains
why the final output equals globally normalized attention.

### 3.3 Trace a large change in the running maximum

The final block below raises the maximum from 4 to 1000. Subtracting the maximum
keeps exponentials at most 1. The old contributions become negligible;
underflow of extremely small weights to zero is harmless here, unlike overflow
of an unshifted exponential. The printed $A$ and $\ell$ share the same scale.

In [8]:
scores_trace = np.array([1., 2., 3., 4., 1000., 999.])
values_trace = np.array([[1., 0.], [0., 1.], [2., 1.], [1., 2.], [4., 0.], [0., 4.]])
m, ell = -np.inf, 0.0
accumulator = np.zeros(2)
print("block     new m        alpha       ell              A              A / ell")
for start in range(0, len(scores_trace), 2):
    scores_block = scores_trace[start:start+2]
    values_block = values_trace[start:start+2]
    m_new = max(m, np.max(scores_block))
    alpha = np.exp(m - m_new)
    weights = np.exp(scores_block - m_new)
    ell = alpha * ell + np.sum(weights)
    accumulator = alpha * accumulator + weights @ values_block
    m = m_new
    print(f"{start//2 + 1:3d} {m:11.1f} {alpha:12.3e} {ell:9.5f}  {accumulator}  {accumulator / ell}")

expected_trace = (softmax_rows(scores_trace[None, :]) @ values_trace)[0]
np.testing.assert_allclose(accumulator / ell, expected_trace, atol=1e-12)
print("Online and global outputs match:", expected_trace)

block     new m        alpha       ell              A              A / ell
  1         2.0    0.000e+00   1.36788  [0.36788 1.     ]  [0.26894 0.73106]
  2         4.0    1.353e-01   1.55300  [1.78555 2.50321]  [1.14974 1.61186]
  3      1000.0    0.000e+00   1.36788  [4.      1.47152]  [2.92423 1.07577]
Online and global outputs match: [2.92423 1.07577]


## 4. FlashAttention forward: many queries, one tile at a time

We now run the same recurrence for a **block of query rows**. Let $I$ select
$B_q$ queries and $J$ select $B_k$ keys. A tile of scores is

$$S_{IJ}=Q_IK_J^\top/\sqrt d\quad\text{with shape }(B_q,B_k).$$

For this query block, $m,\ell$ have shape $(B_q,1)$ and $A$ has shape
$(B_q,d_v)$. The scalar recurrence from module 3 works row by row using
broadcasting. `weights` below are unnormalized exponentials relative to the
**current** maximum; they are not the final probabilities $P_{IJ}$.

```text
For each query block I:
    initialize m, ell, A
    for each key/value block J:
        compute only S[I, J]
        update m, ell, A
        discard the temporary score and weight tile
    write O[I] = A / ell
    write L[I] = m + log(ell)
```

The saved vector $L$ is the row **log-sum-exp**, not the scalar loss $\mathcal L$:

$$L_i=\log\sum_j e^{S_{ij}}=m_i+\log\ell_i.$$

It will let backward recover $P_{ij}=e^{S_{ij}-L_i}$. Only one number per row
is needed. We retain the input arrays and output too; avoiding $P$ does not mean
backward can run without inputs.

In [9]:
def flash_attention_forward(Q, K, V, block_q, block_k, causal=False):
    """Tiled attention. Return O and row log-sum-exp L, without a full P."""
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((N, V.shape[1]), dtype=Q.dtype)
    L = np.zeros(N, dtype=Q.dtype)

    for i0 in range(0, N, block_q):
        i1 = min(i0 + block_q, N)
        Qi = Q[i0:i1]
        m = np.full((i1 - i0, 1), -np.inf)
        ell = np.zeros((i1 - i0, 1))
        accumulator = np.zeros((i1 - i0, V.shape[1]))

        for j0 in range(0, N, block_k):
            if causal and j0 >= i1:
                break  # This and all later key tiles are entirely in the future.
            j1 = min(j0 + block_k, N)
            Kj, Vj = K[j0:j1], V[j0:j1]
            scores = scale * (Qi @ Kj.T)
            if causal:
                allowed = np.arange(i0, i1)[:, None] >= np.arange(j0, j1)[None, :]
                scores = np.where(allowed, scores, -np.inf)

            m_new = np.maximum(m, np.max(scores, axis=1, keepdims=True))
            alpha = np.exp(m - m_new)
            weights = np.exp(scores - m_new)
            ell = alpha * ell + np.sum(weights, axis=1, keepdims=True)
            accumulator = alpha * accumulator + weights @ Vj
            m = m_new

        O[i0:i1] = accumulator / ell
        L[i0:i1] = (m + np.log(ell)).ravel()

    return O, L

### Verify output and saved normalizers

The length 7 is deliberately not divisible by either tile size. Slicing handles
the smaller final tiles. We use $d_v\ne d$ so that an accidental assumption
about equal feature dimensions is visible.

The **verification code** may build dense scores to check $L$. The FlashAttention
function itself does not create or retain a full score or probability matrix
when tile sizes are smaller than $N$.

In [10]:
Q = rng.normal(size=(7, 4))
K = rng.normal(size=(7, 4))
V = rng.normal(size=(7, 3))
O_dense, P_dense = attention_forward(Q, K, V)
O_flash, L_flash = flash_attention_forward(Q, K, V, block_q=3, block_k=2)

np.testing.assert_allclose(O_flash, O_dense, atol=1e-12, rtol=1e-12)
scores_check = Q @ K.T / np.sqrt(Q.shape[1])
max_check = scores_check.max(axis=1)
L_check = max_check + np.log(np.exp(scores_check - max_check[:, None]).sum(axis=1))
np.testing.assert_allclose(L_flash, L_check, atol=1e-12, rtol=1e-12)
print("Maximum forward error:", np.max(np.abs(O_flash - O_dense)))
print("Saved log-normalizers:", L_flash)

Maximum forward error: 1.6653345369377348e-16
Saved log-normalizers: [1.94311 3.97962 3.17692 1.95633 1.11475 1.82465 3.2593 ]


## 5. FlashAttention backward: reconstruct, then differentiate

### 5.1 Reconstruct probabilities using the final normalization

Forward saved $O$ and $L$. Given the inputs again, compute one score tile and
reconstruct

$$P_{IJ}=\exp(S_{IJ}-L_I).$$

The subtraction broadcasts $L_I$ across keys. These are probabilities under
the **whole row's** softmax, not softmax normalized over tile $J$.
In particular, their row sums need not equal 1 until all key tiles are included.

In [11]:
i0, i1, j0, j1 = 0, 3, 2, 4
scores_tile = Q[i0:i1] @ K[j0:j1].T / np.sqrt(Q.shape[1])
P_reconstructed = np.exp(scores_tile - L_flash[i0:i1, None])
np.testing.assert_allclose(P_reconstructed, P_dense[i0:i1, j0:j1], atol=1e-12)
print("Reconstructed probability tile:\n", P_reconstructed)
print("Partial row sums:", P_reconstructed.sum(axis=1))

Reconstructed probability tile:
 [[0.10647 0.16447]
 [0.16654 0.01911]
 [0.50667 0.08035]]
Partial row sums: [0.27094 0.18565 0.58703]


### 5.2 Compute the softmax row correction without a full $dP$

Dense backward needed $D_i=\sum_jP_{ij}\,dP_{ij}$. At first this appears to
require the entire row of $P$ and $dP$. Substitute $dP_{ij}=\sum_rG_{ir}V_{jr}$:

$$
\begin{aligned}
D_i
&=\sum_jP_{ij}\sum_rG_{ir}V_{jr}\\
&=\sum_rG_{ir}\left(\sum_jP_{ij}V_{jr}\right)\\
&=\sum_rG_{ir}O_{ir}.
\end{aligned}
$$

So `D = np.sum(G * O, axis=1, keepdims=True)` needs only the output and its
upstream gradient. It reduces over **value features**, whereas the dense
expression reduces over **keys**. The two sums give the same scalar per query.

In [12]:
G = rng.normal(size=O_dense.shape)
D_from_probabilities = np.sum(P_dense * (G @ V.T), axis=1)
D_from_output = np.sum(G * O_flash, axis=1)
np.testing.assert_allclose(D_from_output, D_from_probabilities, atol=1e-12)
print("Maximum error in the D identity:", np.max(np.abs(D_from_output - D_from_probabilities)))

Maximum error in the D identity: 1.1102230246251565e-16


### 5.3 Use the ordinary derivatives on each tile

For query block $I$ and key/value block $J$:

$$
\begin{aligned}
dP_{IJ}&=G_I V_J^\top,\\
dS_{IJ}&=P_{IJ}\odot(dP_{IJ}-D_I),\\
dV_J&\mathrel{+}=P_{IJ}^\top G_I,\\
dQ_I&\mathrel{+}=dS_{IJ}K_J/\sqrt d,\\
dK_J&\mathrel{+}=dS_{IJ}^\top Q_I/\sqrt d.
\end{aligned}
$$

The `+=` signs matter: $dQ_I$ receives contributions from every key block;
$dK_J$ and $dV_J$ receive contributions from every query block. We use a
key-block outer loop here to make those shared key/value contributions easy
to follow. This is a sequential CPU loop, with no parallel gradient updates.

### Why not differentiate through the running maximum?

We derived a backward formula for the attention function itself. Online
softmax is another way to evaluate that same function. We may therefore use
the ordinary attention derivative with reconstructed probabilities, without
backpropagating through every running maximum, rescaling, and accumulator
update. This is the key to a simple custom backward.

The memory tradeoff is concrete: compute the scores and exponentials again,
instead of saving all probabilities from forward. Do **not** accumulate tiles
in a list; each tile is temporary.

In [13]:
def flash_attention_backward(Q, K, V, O, L, G, block_q, block_k, causal=False):
    """Recompute probability tiles and accumulate gradients of Q, K, and V."""
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    dQ, dK, dV = np.zeros_like(Q), np.zeros_like(K), np.zeros_like(V)
    D = np.sum(G * O, axis=1, keepdims=True)

    for j0 in range(0, N, block_k):
        j1 = min(j0 + block_k, N)
        Kj, Vj = K[j0:j1], V[j0:j1]

        for i0 in range(0, N, block_q):
            i1 = min(i0 + block_q, N)
            if causal and j0 >= i1:
                continue  # All keys in this tile are after all its queries.
            Qi, Gi = Q[i0:i1], G[i0:i1]
            scores = scale * (Qi @ Kj.T)
            if causal:
                allowed = np.arange(i0, i1)[:, None] >= np.arange(j0, j1)[None, :]
                scores = np.where(allowed, scores, -np.inf)

            P_block = np.exp(scores - L[i0:i1, None])
            dP_block = Gi @ Vj.T
            dS_block = P_block * (dP_block - D[i0:i1])

            dV[j0:j1] += P_block.T @ Gi
            dQ[i0:i1] += scale * (dS_block @ Kj)
            dK[j0:j1] += scale * (dS_block.T @ Qi)

    return dQ, dK, dV

### Compare all three input gradients

This checks the actual backward outputs, not just the forward result. Both
implementations receive the same $G$ and their own saved forward quantities.

In [14]:
dense_grads = attention_backward(Q, K, V, P_dense, G)
flash_grads = flash_attention_backward(Q, K, V, O_flash, L_flash, G, 3, 2)
for name, dense, flash in zip(("dQ", "dK", "dV"), dense_grads, flash_grads):
    np.testing.assert_allclose(flash, dense, atol=1e-12, rtol=1e-12)
    print(f"{name}: maximum dense-vs-tiled error = {np.max(np.abs(flash - dense)):.2e}")

dQ: maximum dense-vs-tiled error = 1.11e-16
dK: maximum dense-vs-tiled error = 1.39e-16
dV: maximum dense-vs-tiled error = 1.67e-16


## 6. Causal attention and end-to-end checks

### 6.1 Mask future keys, including in backward

In causal self-attention, query position $i$ may attend only to positions $j\le i$.
The diagonal is allowed. Replace future scores with $-\infty$ before softmax,
so their probabilities and score gradients are zero:

$$
\widetilde S_{ij}=\begin{cases}S_{ij}&j\le i,\\-\infty&j>i.\end{cases}
$$

Dense attention builds a full mask for reference. FlashAttention builds only
the mask for the current tile, using **global token positions**, not positions
within a tile. A tile entirely above the diagonal is skipped. Diagonal-crossing
tiles are masked element by element. Backward reconstructs with the same mask.

In forward, key blocks are visited from left to right. The first key block
contains position 0, which every query may attend to. Thus each row has a finite
running maximum before it encounters any entirely masked portion of a later
tile. This avoids an undefined $-\infty-(-\infty)$ update. Arbitrary masks that
leave a whole row without any allowed key are outside this tutorial's scope.

The dense backward needs no separate `causal` argument: zeros in its saved $P$
already zero out all masked entries of $dS$.

In [15]:
O_causal, P_causal = attention_forward(Q_tiny, K_tiny, V_tiny, causal=True)
O_causal_flash, L_causal = flash_attention_forward(
    Q_tiny, K_tiny, V_tiny, block_q=2, block_k=2, causal=True
)
print("Causal probabilities:\n", P_causal)
np.testing.assert_allclose(np.triu(P_causal, k=1), 0.0)
np.testing.assert_allclose(P_causal.sum(axis=1), 1.0)
np.testing.assert_allclose(O_causal_flash, O_causal, atol=1e-12)
np.testing.assert_allclose(O_causal[0], V_tiny[0])

# A later key/value cannot affect earlier outputs in causal attention.
K_changed, V_changed = K_tiny.copy(), V_tiny.copy()
K_changed[-1] += 10.0
V_changed[-1] -= 10.0
O_changed, _ = flash_attention_forward(Q_tiny, K_changed, V_changed, 2, 2, causal=True)
np.testing.assert_allclose(O_changed[:-1], O_causal_flash[:-1], atol=1e-12)
print("Causal checks passed: future keys/values do not affect earlier outputs.")

Causal probabilities:
 [[1.      0.      0.     ]
 [0.33024 0.66976 0.     ]
 [0.40111 0.40111 0.19778]]
Causal checks passed: future keys/values do not affect earlier outputs.


### 6.2 Compare forward and backward across tile sizes

These small cases test the algorithm rather than production input validation:

- A sequence with just one token.
- Uneven final tiles and different query/key tile sizes.
- One token per tile, which makes all the accumulation steps explicit.
- Tiles larger than the sequence, reducing to a single full tile.
- Both unrestricted and causal attention, with $d_v\ne d$.

We compare at `atol=rtol=1e-12` for these moderate, float64 inputs. This is not
a universal tolerance for other data types or extreme score magnitudes.
The table reports maximum absolute error across all cases in each mode.

In [16]:
check_rng = np.random.default_rng(2026)
tile_sizes = [(1, 1), (2, 3), (3, 2), (4, 4), (16, 16)]
shapes = [(1, 2, 3), (5, 3, 2), (7, 4, 3)]
comparison_cases = 0
print("causal  cases      output          dQ          dK          dV")
for causal in (False, True):
    largest_errors = np.zeros(4)
    cases_this_mode = 0
    for N, d, d_v in shapes:
        q = check_rng.normal(size=(N, d))
        k = check_rng.normal(size=(N, d))
        v = check_rng.normal(size=(N, d_v))
        g = check_rng.normal(size=(N, d_v))
        reference_O, reference_P = attention_forward(q, k, v, causal=causal)
        reference_grads = attention_backward(q, k, v, reference_P, g)
        for block_q, block_k in tile_sizes:
            tiled_O, tiled_L = flash_attention_forward(q, k, v, block_q, block_k, causal=causal)
            tiled_grads = flash_attention_backward(
                q, k, v, tiled_O, tiled_L, g, block_q, block_k, causal=causal
            )
            expected_arrays = (reference_O,) + reference_grads
            actual_arrays = (tiled_O,) + tiled_grads
            for index, (actual, expected) in enumerate(zip(actual_arrays, expected_arrays)):
                np.testing.assert_allclose(actual, expected, atol=1e-12, rtol=1e-12)
                largest_errors[index] = max(largest_errors[index], np.max(np.abs(actual - expected)))
            comparison_cases += 1
            cases_this_mode += 1
    print(f"{str(causal):6s} {cases_this_mode:5d} " + " ".join(f"{e:11.2e}" for e in largest_errors))
print(f"All {comparison_cases} forward/backward comparison cases passed.")

causal  cases      output          dQ          dK          dV
False     15    3.33e-16    4.44e-16    3.61e-16    2.78e-16
True      15    2.22e-16    2.78e-16    3.33e-16    3.89e-16
All 30 forward/backward comparison cases passed.


### 6.3 Finite differences for both implementations, in both modes

Two implementations could share the same backward mistake, so agreement alone
is insufficient. Now check each analytical backward against finite differences
of **its own forward function**. We keep $G$ fixed while perturbing $Q,K,V$ and
recompute forward statistics for every perturbation.

Finite differences subtract nearby numbers and divide by a small step, so we
use looser tolerances: `atol=2e-8`, `rtol=2e-6`, and $\varepsilon=10^{-6}$.
Making the step arbitrarily small can worsen cancellation error.

In [17]:
fd_rng = np.random.default_rng(123)
q_fd = fd_rng.normal(size=(3, 2))
k_fd = fd_rng.normal(size=(3, 2))
v_fd = fd_rng.normal(size=(3, 3))
g_fd = fd_rng.normal(size=(3, 3))
finite_difference_checks = 0
print("method  causal           dQ          dK          dV")
for causal in (False, True):
    dense_o, dense_p = attention_forward(q_fd, k_fd, v_fd, causal=causal)
    flash_o, flash_l = flash_attention_forward(q_fd, k_fd, v_fd, 2, 2, causal=causal)
    dense_analytic = attention_backward(q_fd, k_fd, v_fd, dense_p, g_fd)
    flash_analytic = flash_attention_backward(
        q_fd, k_fd, v_fd, flash_o, flash_l, g_fd, 2, 2, causal=causal
    )
    methods = [
        ("dense", lambda q, k, v: attention_forward(q, k, v, causal=causal)[0], dense_analytic),
        ("flash", lambda q, k, v: flash_attention_forward(q, k, v, 2, 2, causal=causal)[0], flash_analytic),
    ]
    for method, forward, analytical in methods:
        numerical = numerical_gradients(forward, q_fd, k_fd, v_fd, g_fd)
        errors = []
        for analytic, numeric in zip(analytical, numerical):
            np.testing.assert_allclose(analytic, numeric, atol=2e-8, rtol=2e-6)
            errors.append(np.max(np.abs(analytic - numeric)))
            finite_difference_checks += 1
        print(f"{method:7s} {str(causal):6s} " + " ".join(f"{e:11.2e}" for e in errors))
print(f"All {finite_difference_checks} gradient-array checks passed (every element checked).")

method  causal           dQ          dK          dV
dense   False     1.99e-10    8.84e-11    1.65e-10
flash   False     8.41e-11    1.81e-10    1.69e-10
dense   True      3.09e-10    3.13e-10    1.92e-10
flash   True      3.32e-10    1.85e-10    1.82e-10
All 12 gradient-array checks passed (every element checked).


## 7. What memory did we save?

The distinction is between arrays that persist between forward and backward,
and temporary arrays used while processing a tile.

| Quantity | Dense reference | Tiled implementation |
|---|---|---|
| Inputs and output | $Q,K,V,O$ | $Q,K,V,O$ |
| Probability information saved for backward | $P$: $N^2$ numbers | $L$: $N$ numbers |
| Score/probability temporaries | Full $N\times N$ arrays | Tiles of at most $B_q\times B_k$ |
| Forward running state | Dense row normalizers | $m,\ell,A$ for one query block |
| Backward temporaries | Full $dP,dS$ | Tile-sized $dP,dS$, plus row vector $D$ |
| Returned gradients | $dQ,dK,dV$ | $dQ,dK,dV$ |

With fixed feature dimensions and fixed tile sizes, the tiled implementation's
total array storage grows linearly with $N$, including inputs, outputs, and
gradients. Tile workspace includes a constant number of $B_qB_k$ arrays and
the $B_qd_v$ accumulator; it does not grow with the number of tiles. Choosing
tile sizes as large as $N$ naturally loses that memory benefit.

The next table counts selected arrays analytically, **not measured peak process
memory**. It omits common inputs/outputs, gradients, additional simultaneous
tiles, NumPy temporaries, and BLAS workspace. The dense backward can retain $P$
without retaining $O$; the tiled backward specifically needs $O$ as well as $L$.
The quadratic-versus-linear distinction still holds when that output is counted.

In [18]:
bytes_per_number = np.dtype(np.float64).itemsize
tile_q, tile_k = 32, 32
print("    N    saved dense P (KiB)    saved L (KiB)    one score tile (KiB)")
for length in (128, 512, 2048, 8192):
    dense_kib = length * length * bytes_per_number / 1024
    normalizer_kib = length * bytes_per_number / 1024
    tile_kib = min(length, tile_q) * min(length, tile_k) * bytes_per_number / 1024
    print(f"{length:5d} {dense_kib:22.1f} {normalizer_kib:16.1f} {tile_kib:23.1f}")

    N    saved dense P (KiB)    saved L (KiB)    one score tile (KiB)
  128                  128.0              1.0                     8.0
  512                 2048.0              4.0                     8.0
 2048                32768.0             16.0                     8.0
 8192               524288.0             64.0                     8.0


### Why can recomputing be faster on a GPU?

Real FlashAttention keeps small tiles in fast on-chip storage, reducing transfers
to and from slower high-bandwidth memory. Extra arithmetic can be worthwhile
when it avoids expensive memory traffic. The NumPy implementation illustrates
the array dependencies, but it does not control GPU memory placement or fuse
operations into a kernel. Python loops can make it slower than dense NumPy.

Both algorithms still do quadratic work in sequence length:
$O(N^2(d+d_v))$ for dense attention, with the same asymptotic order for tiled
attention and its backward. Causal masking removes about half the pairs, not
the quadratic growth. The algorithm is exact in real arithmetic, with ordinary
floating-point rounding differences in numerical implementations.

## 8. Recap and next steps

### The whole mechanism in four statements

1. **Dense attention** computes $P=\operatorname{softmax}(QK^\top/\sqrt d)$ and $O=PV$.
2. **Tiled forward** keeps $m,\ell,A$ and rescales both $\ell$ and $A$ when the
   maximum changes. It returns $O=A/\ell$ and saves $L=m+\log\ell$.
3. **Tiled backward** reconstructs $P_{IJ}=\exp(S_{IJ}-L_I)$, uses
   $D_i=\sum_rG_{ir}O_{ir}$, and accumulates the ordinary attention gradients.
4. **The saving** comes from not retaining full score/probability/gradient
   matrices. The input gradients and output still have to exist.

### Interface quick reference

| Function | Returns | Information used later |
|---|---|---|
| `attention_forward(Q, K, V, causal=False)` | `O, P` | Backward needs `Q, K, V, P, G` |
| `attention_backward(Q, K, V, P, G)` | `dQ, dK, dV` | Gradients have their input shapes |
| `flash_attention_forward(Q, K, V, block_q, block_k, causal=False)` | `O, L` | Backward needs the inputs, `O, L, G`, and matching mask |
| `flash_attention_backward(Q, K, V, O, L, G, block_q, block_k, causal=False)` | `dQ, dK, dV` | No full attention matrix is saved |

Use positive tile sizes and nonempty, finite float64 arrays with the shapes from
module 1. Keep inputs unchanged between a forward pass and its matching backward.
`G` must have the shape of `O`. These are study functions with explicit input
assumptions, rather than a general-purpose attention API.

### Exercises

1. In module 3's trace, remove `alpha` from the accumulator update. Use moderate
   changing maxima such as `[1, 2, 3, 4, 8, 9]`. Which check fails, and why?
2. Try different tile sizes and verify that outputs and gradients still agree.
   Explain why backward can use different tile sizes from forward when it has $L$.
3. Set every query to zero. Derive the unmasked output (the mean of all values)
   and the causal output (the mean of each allowed prefix) before running them.
4. Re-derive $D_i$ without looking at module 5. Which stored tensor makes it cheap?
5. Add batch and head axes with simple outer loops. Each head/sequence has its
   own $Q,K,V,O,L$ and follows the same equations. No new softmax derivative is needed.

### Further reading

- [Dao et al., FlashAttention (2022)](https://arxiv.org/abs/2205.14135):
  tiling, online normalization, recomputation, and the memory-I/O motivation.
- [Dao, FlashAttention-2 (2023)](https://arxiv.org/abs/2307.08691):
  the unnormalized forward accumulator used here, plus GPU work partitioning
  that this notebook deliberately leaves out. See Algorithms 1 and 2.
- [Milakov and Gimelshein, Online normalizer calculation for softmax (2018)](https://arxiv.org/abs/1805.02867):
  background on the online normalization recurrence.

These implementations adapt the paper equations for readability and explicitly
include $1/\sqrt d$, which some algorithm presentations omit for brevity.

In [19]:
print(f"Study notebook checks passed: {comparison_cases} dense/tiled cases and "
      f"{finite_difference_checks} end-to-end gradient-array checks.")
print("Also checked: tiny dense gradients, softmax Jacobian, online normalization, "
      "log-normalizers, reconstructed probabilities, the D identity, and causality.")

Study notebook checks passed: 30 dense/tiled cases and 12 end-to-end gradient-array checks.
Also checked: tiny dense gradients, softmax Jacobian, online normalization, log-normalizers, reconstructed probabilities, the D identity, and causality.
